# מפגש 24 — מעבדת תלמיד Audited: רגרסיה ליניארית ו־R²
**90 דקות שיעור ו־45 דקות פרויקט.** משימות המקור נשמרו: לנבא `body_mass_g` לפי `bill_length_mm` בלבד, `random_state=17`. שלוש הדקות הראשונות: מדוע מחיר או מסה הם רגרסיה ולא סיווג? השלימו `???` וכתבו הסבר עצמאי אחרי כל שלב. **לא מוצגים ציוני Test או פתרונות מוכנים.**

## משימה 0 — ציפייה וחישוב ידני
מודל דידקטי נתון: ŷ=0.416x−0.366. עבור x=2.5, חשבו ŷ. אם y=0.7, חשבו `residual = y−ŷ`. אין זה מודל שאומן על Penguins.

In [ ]:
toy_prediction=???*???+???
toy_residual=???-toy_prediction
assert abs(toy_prediction-0.674)<1e-12
print("Prediction",toy_prediction,"Residual",toy_residual)

## משימה 1 — טעינה והגדרת בעיה
הנתונים כוללים מדידות במ״מ ובגרמים. כותבים X כ־DataFrame בעל מאפיין אחד, y כ־Series; מסירים רק רשומות ש־body_mass_g שלהן חסר.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score
url="https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df=pd.read_csv(url)
assert len(df)==344
display(df.head(3))

In [ ]:
model_df=df.dropna(subset=[???]).copy()
feature=???
X=model_df[[feature]].copy()
y=model_df[???].copy()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=???,random_state=???)
assert len(model_df)==342
assert X.shape[1]==1
assert set(X_train.index).isdisjoint(set(X_test.index))

## משימה 1ב — מניעת זליגה
לומדים ממוצע השלמת ערכים חסרים **רק ב־Train**. מדוע אסור לחשב אותו מכל Penguins לפני הפיצול?

In [ ]:
training_means=X_train.???()
X_train_ready=X_train.???(training_means)
X_test_ready=X_test.???(training_means)
assert not X_train_ready.isna().any().any()
assert not X_test_ready.isna().any().any()
assert len(X_test_ready)==86

## משימה 2 — לימוד ישר ותחזית
בצעו fit על קבוצת האימון בלבד. פירשו במילים את יחידת השיפוע: גרמים לכל מ״מ. חיזוי עבור 45 מ״מ אינו הסבר סיבתי.

In [ ]:
model=???()
model.???(X_train_ready,y_train)
predictions=model.???(X_test_ready)
w=float(model.coef_[0])
b=float(model.intercept_)
new_penguin=pd.DataFrame({feature:[???]})
new_prediction=model.predict(new_penguin)[0]
assert np.isclose(new_prediction,w*45+b)
print("Slope g/mm:",w,"Intercept g:",b,"45mm prediction g:",new_prediction)

## משימה 3 — שאריות וגרף
השארית מוגדרת `actual−predicted`. שארית חיובית פירושה שהמודל חזה ערך נמוך מדי. ציירו את הישר לפי X ממוינים כדי להימנע מקו בזיגזג.

In [ ]:
line_X=X_train_ready.sort_values(???)
plt.figure()
plt.scatter(X_train_ready[feature],y_train,alpha=.35,label="Train")
plt.scatter(X_test_ready[feature],y_test,alpha=.65,label="Test")
plt.plot(line_X[feature],model.predict(line_X),color="black",label="Fit on Train")
plt.xlabel("Bill length (mm)")
plt.ylabel("Body mass (g)")
plt.legend()
plt.show()
comparison=pd.DataFrame({"actual_g":y_test,"predicted_g":predictions},index=y_test.index)
comparison["residual_g"]=comparison[???]-comparison[???]
assert len(comparison)==86
assert np.allclose(comparison["residual_g"],y_test.to_numpy()-predictions)
display(comparison.head())

## משימה 4 — R² ידני ומדד ספרייה
(R^2=1-\frac{SSE}{SST}). כאן (SSE=\sum_i(y_i-\hat y_i)^2); (SST=\sum_i(y_i-\bar y_{Test})^2). ממוצע Test הוא חלק **מהגדרת המדד**, ולא הכנה שמותר להשתמש בה בעת אימון.

In [ ]:
sse=(comparison["residual_g"]**2).???()
sst=((y_test-y_test.mean())**2).???()
r2_manual=1-???/???
r2_lib=r2_score(???,???)
assert sst>0
assert np.isclose(r2_manual,r2_lib)
print("SSE",sse,"SST",sst,"R²",r2_lib)

## משימה 4ב — קו בסיס
מאמנים `DummyRegressor` שמחזיר ממוצע מ־Train בלבד. האם R² שלו חייב להיות אפס על Test? נמקו.

In [ ]:
baseline=DummyRegressor(strategy=???)
baseline.fit(???,???)
baseline_pred=baseline.predict(???)
baseline_r2=r2_score(y_test,baseline_pred)
assert np.isclose(baseline.constant_.ravel()[0],y_train.mean())
assert len(baseline_pred)==len(y_test)
print("Baseline R²:",baseline_r2)

## משימה 5 — דוח פרויקט (ללא קוד חדש)
כתבו מהו המשתנה המנובא ומהן יחידותיו; מה משמעות שיפוע וחיתוך; תוצאת תחזית 45 מ״מ; דוגמת שארית חיובית/שלילית; R² לעומת קו בסיס; שתי מגבלות. למה `Accuracy` אינו מדד מתאים? אין להשוות למודל הסנפיר של המורה כראיה להעדפת מאפיין — הזרע והמאפיין שונים.